<a href="https://colab.research.google.com/github/Likith-Reddy25/Summer-Intern/blob/main/codes/Ad_hoc_COV_Covariant_Map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Covariant Map QKE

In [ ]:
!pip install -q qiskit==1.1.0 qiskit-machine-learning==0.7.2 qiskit-algorithms==0.3.0 --no-deps
!pip install -q qiskit-aer==0.14.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.8/97.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.6/308.6 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 MB 16.6 MB/s eta 0:00:00


In [ ]:


import os
import numpy as np
import pandas as pd

from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score
from sklearn.preprocessing import MinMaxScaler

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit_machine_learning.kernels import FidelityStatevectorKernel


def data_map_D(n_qubits: int, x_params: ParameterVector) -> QuantumCircuit:
    """D(x) = kron_k RX(x_{2k-1}) RZ(x_{2k})   (eq. 16)."""
    qc = QuantumCircuit(n_qubits, name="D(x)")
    for k in range(n_qubits):
        qc.rx(x_params[2 * k], k)
        qc.rz(x_params[2 * k + 1], k)
    return qc


def fiducial_state_V(n_qubits: int, theta_params: ParameterVector,
                      strategy: str = "shared") -> QuantumCircuit:

    qc = QuantumCircuit(n_qubits, name="V(theta)")

    def rxyz(circ, q, t1, t2, t3):
        # matches eq. 22: RXYZ(t1,t2,t3) = U3-type gate with polar angle t1,
        # and phases t2 (as "lambda"), t3 (as "phi")
        circ.rz(t3, q)
        circ.ry(t1, q)
        circ.rz(t2, q)

    if strategy == "shared":
        assert len(theta_params) == 3
        for k in range(n_qubits):
            rxyz(qc, k, theta_params[0], theta_params[1], theta_params[2])
    elif strategy == "dedicated":
        assert len(theta_params) == 3 * n_qubits
        for k in range(n_qubits):
            rxyz(qc, k,
                 theta_params[3 * k], theta_params[3 * k + 1], theta_params[3 * k + 2])
    else:
        raise ValueError("strategy must be 'shared' or 'dedicated'")

    # linear entangling layer: CZ_{k,k+1}
    for k in range(n_qubits - 1):
        qc.cz(k, k + 1)

    return qc


def build_covariant_feature_map(n_qubits: int, strategy: str = "shared",
                                 theta_value=0.0) -> QuantumCircuit:
    """
    Full encoding circuit, CORRECT ORDER: V(theta) first (fiducial state
    prep on |0>^n), D(x) second (data-dependent unitary). theta is bound
    to `theta_value` (0.0 => baseline/plain QKE, no kernel training).
    """
    n_features = 2 * n_qubits
    x = ParameterVector("x", n_features)
    n_theta = 3 if strategy == "shared" else 3 * n_qubits
    theta = ParameterVector("theta", n_theta)

    qc = QuantumCircuit(n_qubits, name="U_COV")
    qc.compose(fiducial_state_V(n_qubits, theta, strategy=strategy), inplace=True)
    qc.compose(data_map_D(n_qubits, x), inplace=True)

    if np.isscalar(theta_value):
        bind = {p: theta_value for p in theta}
    else:
        bind = {p: v for p, v in zip(theta, theta_value)}
    qc = qc.assign_parameters(bind)
    return qc, x


# --------------------------------------------------------------------------
# 2. Dataset: Ad-hoc-COV (7-qubit LCE problem)
# --------------------------------------------------------------------------

def load_adhoc_cov_pool(data_dir: str = "./data", filename: str = "dataset_graph7.csv"):

    path = os.path.join(data_dir, filename)
    if not os.path.exists(path):
        return None
    data = np.loadtxt(path, delimiter=",")
    X_pool, y_pool = data[:, :-1], data[:, -1]
    return X_pool, y_pool


def resample_partitions(X_pool, y_pool, n_per_class_train=32, n_per_class_test=32, seed=None):

    rng = np.random.default_rng(seed)
    X_train, y_train, X_test, y_test = [], [], [], []
    for label in np.unique(y_pool):
        idx = np.where(y_pool == label)[0]
        rng.shuffle(idx)
        n_needed = n_per_class_train + n_per_class_test
        if len(idx) < n_needed:
            raise ValueError(
                f"Pool has only {len(idx)} samples for class {label}, "
                f"need {n_needed} (train {n_per_class_train} + test {n_per_class_test}).")
        train_idx = idx[:n_per_class_train]
        test_idx = idx[n_per_class_train:n_needed]
        X_train.append(X_pool[train_idx]); y_train.append(y_pool[train_idx])
        X_test.append(X_pool[test_idx]); y_test.append(y_pool[test_idx])

    X_train, y_train = np.vstack(X_train), np.concatenate(y_train)
    X_test, y_test = np.vstack(X_test), np.concatenate(y_test)

    perm_tr = rng.permutation(len(y_train)); perm_ts = rng.permutation(len(y_test))
    return X_train[perm_tr], y_train[perm_tr], X_test[perm_ts], y_test[perm_ts]


def generate_synthetic_lce(n_qubits=7, n_per_class=32, epsilon=0.1, seed=None):

    rng = np.random.default_rng(seed)
    n_features = 2 * n_qubits

    theta_star = rng.uniform(0, 2 * np.pi, size=3)  # secret "good" shared theta
    feature_map, x_params = build_covariant_feature_map(
        n_qubits, strategy="shared", theta_value=theta_star)
    kernel = FidelityStatevectorKernel(feature_map=feature_map)

    proto_A = rng.uniform(0, 2 * np.pi, size=n_features)
    proto_B = rng.uniform(0, 2 * np.pi, size=n_features)
    protos = np.vstack([proto_A, proto_B])

    def sample_split(n_per_cls, pool_multiplier=8):
        X, y = [], []
        for cls_idx, label in enumerate((-1, 1)):
            count = 0
            while count < n_per_cls:
                batch = rng.uniform(0, 2 * np.pi, size=(pool_multiplier * n_per_cls, n_features))
                K = kernel.evaluate(x_vec=batch, y_vec=protos)  # (N,2) fidelities
                true_label = np.where(K[:, 1] > K[:, 0], 1, -1)
                mask = true_label == label
                take = batch[mask]
                for row in take:
                    if count >= n_per_cls:
                        break
                    y_obs = label if rng.random() > epsilon else -label
                    X.append(row)
                    y.append(y_obs)
                    count += 1
        X, y = np.array(X), np.array(y)
        perm = rng.permutation(len(y))
        return X[perm], y[perm]

    X_train, y_train = sample_split(n_per_class)
    X_test, y_test = sample_split(n_per_class)
    return X_train, y_train, X_test, y_test


def get_adhoc_cov_data(data_dir="./data", seed=None,
                        n_per_class_train=32, n_per_class_test=32):
    pool = load_adhoc_cov_pool(data_dir)
    if pool is not None:
        X_pool, y_pool = pool
        return resample_partitions(X_pool, y_pool,
                                    n_per_class_train=n_per_class_train,
                                    n_per_class_test=n_per_class_test,
                                    seed=seed)
    print("[info] dataset_graph7.csv not found in "
          f"'{data_dir}' - using quantum-kernel-consistent synthetic "
          "fallback data (will not match paper's Table 2 numbers exactly).")
    return generate_synthetic_lce(seed=seed, n_per_class=n_per_class_train)



def scale_to_2pi(X_train, X_test):
    scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    return X_train_s, X_test_s


# --------------------------------------------------------------------------
# 4. Quantum kernel matrix via statevector-simulated fidelity  (eq. 8)
# --------------------------------------------------------------------------

def compute_kernel(X1, X2, feature_map_circuit, bandwidth):
    """Evaluate the fidelity kernel after bandwidth (lambda) rescaling."""
    kernel = FidelityStatevectorKernel(feature_map=feature_map_circuit)
    return kernel.evaluate(x_vec=bandwidth * X1, y_vec=bandwidth * X2)


# --------------------------------------------------------------------------
# 5. Hyperparameter grid + 5-fold CV model selection  (Section III-C)
# --------------------------------------------------------------------------

C_GRID = [0.01, 0.1, 1, 10, 100]
LAMBDA_GRID = [0.001, 0.01, 0.1, 0.5, 1.0]


def select_best_hyperparams(X_train, y_train, feature_map_circuit):
    best_score, best_C, best_lambda = -np.inf, C_GRID[0], LAMBDA_GRID[-1]
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

    for lam in LAMBDA_GRID:
        K_full = compute_kernel(X_train, X_train, feature_map_circuit, lam)
        for C in C_GRID:
            fold_scores = []
            for tr_idx, val_idx in skf.split(X_train, y_train):
                K_tr = K_full[np.ix_(tr_idx, tr_idx)]
                K_val = K_full[np.ix_(val_idx, tr_idx)]
                clf = SVC(C=C, kernel="precomputed")
                clf.fit(K_tr, y_train[tr_idx])
                pred = clf.predict(K_val)
                fold_scores.append(accuracy_score(y_train[val_idx], pred))
            mean_acc = np.mean(fold_scores)
            if mean_acc > best_score:
                best_score, best_C, best_lambda = mean_acc, C, lam

    return best_C, best_lambda


# --------------------------------------------------------------------------
# 6. Main experiment loop: n = 30 repetitions                 (Section IV)
# --------------------------------------------------------------------------

def run_experiment(n_qubits=7, n_repetitions=30, strategy="shared",
                    data_dir="./data", seed0=0, verbose=True):
    feature_map_circuit, x_params = build_covariant_feature_map(
        n_qubits, strategy=strategy, theta_value=0.0)

    metrics = {"accuracy_TR": [], "kappa_TR": [], "f1_TR": [],
               "accuracy_TS": [], "kappa_TS": [], "f1_TS": []}

    for rep in range(n_repetitions):
        X_train, y_train, X_test, y_test = get_adhoc_cov_data(
            data_dir=data_dir, seed=seed0 + rep)
        X_train, X_test = scale_to_2pi(X_train, X_test)

        best_C, best_lambda = select_best_hyperparams(X_train, y_train, feature_map_circuit)

        K_train = compute_kernel(X_train, X_train, feature_map_circuit, best_lambda)
        K_test = compute_kernel(X_test, X_train, feature_map_circuit, best_lambda)

        if verbose and rep == 0:
            print(f"[sanity check] kernel diag mean (should be ~1.0): "
                  f"{np.mean(np.diag(K_train)):.4f}")

        clf = SVC(C=best_C, kernel="precomputed")
        clf.fit(K_train, y_train)

        pred_train = clf.predict(K_train)
        pred_test = clf.predict(K_test)

        metrics["accuracy_TR"].append(accuracy_score(y_train, pred_train))
        metrics["kappa_TR"].append(cohen_kappa_score(y_train, pred_train))
        metrics["f1_TR"].append(f1_score(y_train, pred_train, average="macro"))
        metrics["accuracy_TS"].append(accuracy_score(y_test, pred_test))
        metrics["kappa_TS"].append(cohen_kappa_score(y_test, pred_test))
        metrics["f1_TS"].append(f1_score(y_test, pred_test, average="macro"))

        print(f"[rep {rep + 1:02d}/{n_repetitions}] "
              f"C={best_C}, lambda={best_lambda}, "
              f"acc_TR={metrics['accuracy_TR'][-1]:.3f}, "
              f"acc_TS={metrics['accuracy_TS'][-1]:.3f}")

    return metrics


def summarize_table2_row(metrics, dataset="Ad-hoc-COV", feature_map="CovariantFeatureMap"):
    """Format results the same way as paper Table 2: mean (std)."""
    row = {"Dataset": dataset, "Feature map": feature_map, "QKT strategy": "-- (plain QKE)"}
    for key in ["accuracy_TR", "kappa_TR", "f1_TR",
                "accuracy_TS", "kappa_TS", "f1_TS"]:
        arr = np.array(metrics[key])
        row[key] = f"{arr.mean():.3f} ({arr.std():.3f})"
    return pd.DataFrame([row])


if __name__ == "__main__":
    results = run_experiment(n_qubits=7, n_repetitions=30, strategy="shared",
                              data_dir=r"C:\Users\likit\Downloads\dataset_graph7.csv", seed0=0)
    table = summarize_table2_row(results)
    print("\n=== Table 2 style summary (qSVM_COV, Ad-hoc-COV) ===")
    print(table.to_string(index=False))
    table.to_csv("qsvm_cov_adhoc_results.csv", index=False)

[info] dataset_graph7.csv not found in 'C:\Users\likit\Downloads\dataset_graph7.csv' - using quantum-kernel-consistent synthetic fallback data (will not match paper's Table 2 numbers exactly).
[sanity check] kernel diag mean (should be ~1.0): 1.0000
[rep 01/30] C=10, lambda=0.1, acc_TR=0.719, acc_TS=0.469
[info] dataset_graph7.csv not found in 'C:\Users\likit\Downloads\dataset_graph7.csv' - using quantum-kernel-consistent synthetic fallback data (will not match paper's Table 2 numbers exactly).
[rep 02/30] C=1, lambda=0.5, acc_TR=1.000, acc_TS=0.516
[info] dataset_graph7.csv not found in 'C:\Users\likit\Downloads\dataset_graph7.csv' - using quantum-kernel-consistent synthetic fallback data (will not match paper's Table 2 numbers exactly).
[rep 03/30] C=1, lambda=0.5, acc_TR=1.000, acc_TS=0.500
[info] dataset_graph7.csv not found in 'C:\Users\likit\Downloads\dataset_graph7.csv' - using quantum-kernel-consistent synthetic fallback data (will not match paper's Table 2 numbers exactly).
[r

QKT shared 3

In [ ]:
!pip install qiskit qiskit-machine-learning qiskit-algorithms scikit-learn numpy pandas
python qkt_zz_shared3_selfcontained.py

SyntaxError: invalid syntax (698857406.py, line 2)